# New Dataset Preparation - Expanded

### Packages

In [1]:
# Basic Package Imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# sklearn
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import RandomForestClassifier
from sklearn.decomposition import PCA
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from sklearn.metrics import f1_score
from sklearn.metrics import mean_squared_error, r2_score

# imblearn
from imblearn.over_sampling import RandomOverSampler
from imblearn.under_sampling import RandomUnderSampler

# Non-basic package imports
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
import requests

# Packages I don't understand
from fcd_torch import FCD
import rdkit
from collections import Counter
import gc
import pickle

# Add the Python_files directory to the Python path
import sys
import os
sys.path.append(os.path.join(os.path.dirname(os.getcwd()), 'Python_files'))

# Now you can import your modules
# import functions_enc as f
import function_depot as fd

# Data Processing

=== GROUP STATISTICS ===
Group                     Total Spectra   Unique SMILES  
-------------------------------------------------------
LTQ-Orbitrap-negative     121             23             
LTQ-Orbitrap-positive     615             70             
LTQ-negative              13              10             
LTQ-positive              12              2              
Other-negative            12              8              
Other-positive            79              25             
Q-Orbitrap-negative       1400            252            
Q-Orbitrap-positive       2065            378            
Q-TOF-negative            235             56             
Q-TOF-positive            1098            221            
QQQ-negative              162             34             
QQQ-positive              286             85             

Total across all groups: 6098            678      

The code that produced this

Print group statistics for unique SMILES and spectra counts
print("=== GROUP STATISTICS ===")
print(f"{'Group':<25} {'Total Spectra':<15} {'Unique SMILES':<15}")
print("-" * 55)

for group in sorted(df5['Group'].unique()):
    group_data = df5[df5['Group'] == group]
    total_spectra = len(group_data)
    unique_smiles = group_data['SMILES_spectra'].nunique()
    
    print(f"{group:<25} {total_spectra:<15} {unique_smiles:<15}")

print(f"\nTotal across all groups: {len(df5):<15} {df5['SMILES_spectra'].nunique():<15}")

Groups included: ['Q-Orbitrap-positive' 'Q-TOF-positive' 'LTQ-Orbitrap-positive']
Unique SMILES: 485

In [2]:
# We are working with the June 25 dataset, with the Morgan Fingerprints and cannonical SMILES included
df5 = pd.read_csv("/home/dlipsey/MITLincolnLabs/MIT_LL_data/MIT_LL_data5.csv")
# print(df5.shape)
# df5.head()

# First order of business is to standardize our SMILES column. We want to use canonical smiles rather than SMILES_spectra but 
# we will keep the column name SMILES_spectra for consistency with previous code
df5 = df5.drop('SMILES_spectra', axis=1) # Drop
df5 = df5.rename(columns={'canonical_smiles': 'SMILES_spectra'}) # Rename
cols = df5.columns.tolist()
cols.remove('SMILES_spectra') 
df5 = df5[['SMILES_spectra'] + cols] # Move to front

# Next we want to standardize the Ionization column
# print(df5["Ionization_Mode"].unique()) # Check unique values
df5["Ionization_Mode"] = df5["Ionization_Mode"].replace("'Positive'", "'positive'") # Fix capitaliztion
df5 = df5[df5["Ionization_Mode"] != "'N/A'"] # Remove N/A 
# print(df5["Ionization_Mode"].unique()) # Check unique values

# Remove single quotes from all columns
df5 = df5.applymap(lambda x: x.replace("'", "") if isinstance(x, str) else x)

# Select specific groups for subset
selected_groups = ['Q-Orbitrap-positive', 'Q-TOF-positive', 'LTQ-Orbitrap-positive']

# Create subset with only selected groups
df5_subset = df5[df5['Group'].isin(selected_groups)]

print(df5_subset.shape)
df5_subset.head()

(3778, 18)


,SMILES_spectra,CAS,Molecular_Formula,Total_Exact_Mass,Precursor_m/z,Precursor_Type,Spectrum,Ionization_Mode,Instrument_Type,Instrument_Name,Collision_Energy,SMILES_tox_vals,Response_Modifier,Response,Response_Unit,Group,fp,filtered_fp
0,C#CCN(C)Cc1ccccc1,555-57-7,C11H13N,159.104799416,160.1121,[M+H]+,63.0228:0.177223 65.0386:5.629055 68.0495:0.45...,positive,LC-ESI-QFT,Q Exactive Orbitrap Thermo Scientific,90 (nominal),C#CCN(C)Cc1ccccc1,,273.642508,mg/kg,Q-Orbitrap-positive,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
1,C#CCN(C)Cc1ccccc1,555-57-7,C11H13N,159.104799416,160.1121,[M+H]+,63.0228:0.125979 65.0386:2.113734 68.0495:0.68...,positive,LC-ESI-QFT,Q Exactive Orbitrap Thermo Scientific,75 (nominal),C#CCN(C)Cc1ccccc1,,273.642508,mg/kg,Q-Orbitrap-positive,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
2,C#CCN(C)Cc1ccccc1,555-57-7,C11H13N,159.104799416,160.1121,[M+H]+,56.0496:0.115017 65.0386:0.970445 68.0495:1.03...,positive,LC-ESI-QFT,Q Exactive Orbitrap Thermo Scientific,60 (nominal),C#CCN(C)Cc1ccccc1,,273.642508,mg/kg,Q-Orbitrap-positive,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
3,C#CCN(C)Cc1ccccc1,555-57-7,C11H13N,159.104799416,160.1121,[M+H]+,51.0229:0.102992 56.0495:0.143820 65.0386:0.67...,positive,LC-ESI-QFT,Q Exactive Orbitrap Thermo Scientific,45 (nominal),C#CCN(C)Cc1ccccc1,,273.642508,mg/kg,Q-Orbitrap-positive,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
4,C#CCN(C)Cc1ccccc1,555-57-7,C11H13N,159.104799416,160.1121,[M+H]+,56.0496:0.482623 65.0385:0.377829 68.0495:2.59...,positive,LC-ESI-QFT,Q Exactive Orbitrap Thermo Scientific,30 (nominal),C#CCN(C)Cc1ccccc1,,273.642508,mg/kg,Q-Orbitrap-positive,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."


In [3]:
# Save the subset dataframe
# df5_subset.to_csv('/home/dlipsey/MITLincolnLabs/MIT_LL_data/df5_subset.csv', index=False)

print(f"Saved subset with {df5_subset.shape[0]} rows and {df5_subset.shape[1]} columns")
print(f"Groups included: {df5_subset['Group'].unique()}")
print(f"Unique SMILES: {df5_subset['SMILES_spectra'].nunique()}")

import os
file_path = '/home/dlipsey/MITLincolnLabs/MIT_LL_data/df5_subset.csv'
if os.path.exists(file_path):
    print(f"File exists at: {file_path}")
    print(f"File size: {os.path.getsize(file_path)} bytes")
else:
    print(f"File does NOT exist at: {file_path}")

Saved subset with 3778 rows and 18 columns
Groups included: ['Q-Orbitrap-positive' 'Q-TOF-positive' 'LTQ-Orbitrap-positive']
Unique SMILES: 485
File exists at: /home/dlipsey/MITLincolnLabs/MIT_LL_data/df5_subset.csv
File size: 49439399 bytes


In [4]:
# Create dataframe with spectra using spectrum_string_to_dataframe
df5_spectra = fd.spectrum_string_to_dataframe(df5_subset, spectrum_col='Spectrum', smiles_col='SMILES_spectra')

print("=== SPECTRA DATAFRAME ===")
print(f"Shape: {df5_spectra.shape}")
print(f"Unique SMILES: {df5_spectra['SMILES_spectra'].nunique()}")
df5_spectra.head()

=== SPECTRA DATAFRAME ===
Shape: (3778, 64441)
Unique SMILES: 485


,SMILES_spectra,29.0112,30.032,30.0323,31.01686,31.54035,38.5076,39.0214,39.0215,39.02194,...,1881.465942,1890.796509,1894.779541,1939.240845,1965.805054,1966.380615,1982.848389,2000.461914,2000.942627,index_id
0,C#CCN(C)Cc1ccccc1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0
1,C#CCN(C)Cc1ccccc1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1
2,C#CCN(C)Cc1ccccc1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2
3,C#CCN(C)Cc1ccccc1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,3
4,C#CCN(C)Cc1ccccc1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,4


In [28]:
# Create dataframe with just SMILES and Morgan fingerprints using expand_fingerprints_to_matrix
df5_morganfp= fd.expand_fingerprints_to_matrix(df5_subset, smiles_col='SMILES_spectra', fp_col='fp')

print("\n=== FINGERPRINTS DATAFRAME ===")
print(f"Shape: {df5_morganfp.shape}")
print(f"Unique SMILES: {df5_morganfp['SMILES_spectra'].nunique()}")
df5_morganfp.head()

Created matrix with 3778 rows and 2048 fingerprint bits
Shape: (3778, 2049)

=== FINGERPRINTS DATAFRAME ===
Shape: (3778, 2049)
Unique SMILES: 485


,SMILES_spectra,bit_1,bit_2,bit_3,bit_4,bit_5,bit_6,bit_7,bit_8,bit_9,...,bit_2039,bit_2040,bit_2041,bit_2042,bit_2043,bit_2044,bit_2045,bit_2046,bit_2047,bit_2048
0,C#CCN(C)Cc1ccccc1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,C#CCN(C)Cc1ccccc1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,C#CCN(C)Cc1ccccc1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,C#CCN(C)Cc1ccccc1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,C#CCN(C)Cc1ccccc1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [30]:
print(type(df5_morganfp))
df5_morganfp = pd.DataFrame(df5_morganfp)

<class 'pandas.core.frame.DataFrame'>


In [42]:
# Create ChemNet embeddings dataframe using get_chemnet_emb_from_smiles
unique_smiles = df5_subset['SMILES_spectra'].unique().tolist()
print(f"Getting ChemNet embeddings for {len(unique_smiles)} unique SMILES...")

# Get embeddings dictionary
embeddings_dict = fd.get_chemnet_emb_from_smiles(unique_smiles)

# Convert to dataframe format
embeddings_data = []
for smiles, embedding in embeddings_dict.items():
    if embedding != 'unknown':  # Skip unknown embeddings
        row = {'SMILES_spectra': smiles}
        # Add each embedding dimension as a separate column
        for i, emb_val in enumerate(embedding):
            row[f'Embedding Float {i}'] = emb_val
        embeddings_data.append(row)

df5_chemnet = pd.DataFrame(embeddings_data)

print("\n=== EMBEDDINGS DATAFRAME ===")
print(f"Shape: {df5_chemnet.shape}")
print(f"Unique SMILES: {df5_chemnet['SMILES_spectra'].nunique()}")
print(f"Embedding dimensions: {df5_chemnet.shape[1] - 1}")  # -1 for SMILES column
df5_chemnet.head()


Getting ChemNet embeddings for 485 unique SMILES...


/home/dlipsey/MITLincolnLabs/.venv/lib/python3.8/site-packages/fcd_torch/fcd.py:51: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  keras_config = torch.load(model_path)



=== EMBEDDINGS DATAFRAME ===
Shape: (485, 513)
Unique SMILES: 485
Embedding dimensions: 512


,SMILES_spectra,Embedding Float 0,Embedding Float 1,Embedding Float 2,Embedding Float 3,Embedding Float 4,Embedding Float 5,Embedding Float 6,Embedding Float 7,Embedding Float 8,...,Embedding Float 502,Embedding Float 503,Embedding Float 504,Embedding Float 505,Embedding Float 506,Embedding Float 507,Embedding Float 508,Embedding Float 509,Embedding Float 510,Embedding Float 511
0,C#CCN(C)Cc1ccccc1,-0.081878,-0.000592,-0.115424,0.361975,-0.013771,0.068946,-0.079094,-0.639643,-0.616823,...,0.120262,0.757480,0.757137,-1.207158e-04,-0.201007,-0.281432,-0.574846,0.037261,0.048509,0.227904
1,c1ccc(CNc2[nH]cnc3ncnc2-3)cc1,0.276912,0.001158,0.184940,0.435511,-0.044371,0.079663,-0.325618,-0.039556,-0.027167,...,0.008954,-0.361290,0.692838,-6.056217e-04,-0.156619,-0.154309,0.284962,0.479151,0.132179,0.100753
2,c1coc(CNc2[nH]cnc3ncnc2-3)c1,0.343482,0.000665,0.109500,0.623262,-0.109757,0.112946,-0.418121,-0.035453,-0.004837,...,0.001494,-0.356037,0.435746,-1.227337e-03,0.135886,0.161948,0.331081,0.624193,0.150636,0.398498
3,c1ccc(CNc2ncnc3c2ncn3C2CCCCO2)cc1,-0.178223,0.005416,0.428077,0.479779,-0.162642,0.033273,-0.322550,-0.034872,-0.049524,...,0.002964,0.204067,0.215919,-1.736632e-02,0.542047,0.477724,-0.102496,0.484719,0.349642,-0.029105
4,c1cc(-c2ccncc2)ccn1,0.340481,0.000491,-0.096004,0.602171,-0.004242,0.101546,0.200853,-0.061073,-0.239671,...,0.034536,-0.191667,0.951767,8.522607e-07,-0.169929,-0.143720,-0.217218,-0.283479,0.141238,0.285142


In [39]:
df5_chemnet = df5_chemnet.to_csv('/home/dlipsey/MITLincolnLabs/MIT_LL_data/df5_chemnet.csv', index=False)
df = pd.read_csv('/home/dlipsey/MITLincolnLabs/MIT_LL_data/df5_chemnet.csv')
print(df.shape) 
df.head()

(485, 513)


,SMILES_spectra,Embedding Float 0,Embedding Float 1,Embedding Float 2,Embedding Float 3,Embedding Float 4,Embedding Float 5,Embedding Float 6,Embedding Float 7,Embedding Float 8,...,Embedding Float 502,Embedding Float 503,Embedding Float 504,Embedding Float 505,Embedding Float 506,Embedding Float 507,Embedding Float 508,Embedding Float 509,Embedding Float 510,Embedding Float 511
0,C#CCN(C)Cc1ccccc1,-0.081878,-0.000592,-0.115424,0.361975,-0.013771,0.068946,-0.079094,-0.639643,-0.616823,...,0.120262,0.757480,0.757136,-1.207158e-04,-0.201007,-0.281432,-0.574846,0.037261,0.048509,0.227904
1,c1ccc(CNc2[nH]cnc3ncnc2-3)cc1,0.276912,0.001158,0.184940,0.435511,-0.044371,0.079663,-0.325618,-0.039556,-0.027167,...,0.008954,-0.361290,0.692838,-6.056217e-04,-0.156619,-0.154309,0.284962,0.479151,0.132179,0.100753
2,c1coc(CNc2[nH]cnc3ncnc2-3)c1,0.343482,0.000665,0.109500,0.623262,-0.109757,0.112946,-0.418121,-0.035453,-0.004837,...,0.001494,-0.356037,0.435746,-1.227337e-03,0.135886,0.161948,0.331081,0.624193,0.150636,0.398498
3,c1ccc(CNc2ncnc3c2ncn3C2CCCCO2)cc1,-0.178223,0.005416,0.428077,0.479779,-0.162642,0.033273,-0.322550,-0.034872,-0.049524,...,0.002964,0.204067,0.215919,-1.736632e-02,0.542047,0.477724,-0.102496,0.484719,0.349642,-0.029105
4,c1cc(-c2ccncc2)ccn1,0.340481,0.000491,-0.096004,0.602171,-0.004242,0.101546,0.200853,-0.061073,-0.239672,...,0.034536,-0.191667,0.951767,8.522606e-07,-0.169929,-0.143720,-0.217218,-0.283479,0.141238,0.285142


In [40]:
df5_morganfp = pd.read_csv('/home/dlipsey/MITLincolnLabs/MIT_LL_data/df5_morganfp.csv')
print(df5_morganfp.shape) 
df5_morganfp.head()

(3778, 2049)


,SMILES_spectra,bit_1,bit_2,bit_3,bit_4,bit_5,bit_6,bit_7,bit_8,bit_9,...,bit_2039,bit_2040,bit_2041,bit_2042,bit_2043,bit_2044,bit_2045,bit_2046,bit_2047,bit_2048
0,C#CCN(C)Cc1ccccc1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,C#CCN(C)Cc1ccccc1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,C#CCN(C)Cc1ccccc1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,C#CCN(C)Cc1ccccc1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,C#CCN(C)Cc1ccccc1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [43]:
df5_chemnet = df5_chemnet.to_csv('/home/dlipsey/MITLincolnLabs/MIT_LL_data/df5_chemnet.csv', index=False)


In [44]:
# # Save the spectra dataframe
# df5_spectra.to_csv('/home/dlipsey/MITLincolnLabs/MIT_LL_data/df5_spectra.csv', index=False)

# print(f"Saved spectra dataframe with {df5_spectra.shape[0]} rows and {df5_spectra.shape[1]} columns")
# print(f"File saved as: df5_spectra.csv")
# print(f"Unique SMILES: {df5_spectra['SMILES_spectra'].nunique()}")

# # Verify the file was created
# import os
# file_path = '/home/dlipsey/MITLincolnLabs/MIT_LL_data/df5_spectra.csv'
# if os.path.exists(file_path):
#     print(f"File exists at: {file_path}")
#     print(f"File size: {os.path.getsize(file_path)} bytes")
# else:
#     print(f"File does NOT exist at: {file_path}")
# Save the fingerprints dataframe

# df5_morganfp = df5_morganfp.to_csv('/home/dlipsey/MITLincolnLabs/MIT_LL_data/df5_morganfp.csv', index=False)

# Verify the file was created
import os
file_path = '/home/dlipsey/MITLincolnLabs/MIT_LL_data/df5_morganfp.csv'
if os.path.exists(file_path):
    print(f"File exists at: {file_path}")
    print(f"File size: {os.path.getsize(file_path)} bytes")
else:
    print(f"File does NOT exist at: {file_path}")
    
# Save the ChemNet embeddings dataframe
# df5_chemnet = df5_chemnet.to_csv('/home/dlipsey/MITLincolnLabs/MIT_LL_data/df5_chemnet.csv', index=False)


# Verify the file was created
import os
file_path = '/home/dlipsey/MITLincolnLabs/MIT_LL_data/df5_chemnet.csv'
if os.path.exists(file_path):
    print(f"File exists at: {file_path}")
    print(f"File size: {os.path.getsize(file_path)} bytes")
else:
    print(f"File does NOT exist at: {file_path}")

File exists at: /home/dlipsey/MITLincolnLabs/MIT_LL_data/df5_morganfp.csv
File size: 15614639 bytes
File exists at: /home/dlipsey/MITLincolnLabs/MIT_LL_data/df5_chemnet.csv
File size: 2991834 bytes
